# Yojana Sahayak — QLoRA Fine-tuning
**Model:** Qwen2.5-1.5B-Instruct  
**Dataset:** Subh24ai/yojana-sahayak-instruct  
**Task:** Hindi + English Q&A on Indian government schemes  
**Hardware:** Kaggle T4 GPU (16GB VRAM)  

Pipeline:
1. Load dataset from HuggingFace Hub
2. Load Qwen2.5-1.5B in 4-bit (BitsAndBytes)
3. Attach LoRA adapters (r=16, alpha=32)
4. Train with SFTTrainer
5. Evaluate: ROUGE + loss curve
6. Push adapter + merged model to HF Hub

## Step 1 — Install dependencies

In [ ]:
%%capture
!pip install -q transformers==4.46.3 datasets peft tqdm
!pip install -q trl==0.12.1 bitsandbytes accelerate
!pip install -q rouge-score wandb huggingface_hub
print('Installation complete')

## Step 2 — Config (edit here only)

In [ ]:
# ── EDIT THESE ────────────────────────────────────────────────
HF_TOKEN      = "hf_YOUR_TOKEN_HERE"   # huggingface.co/settings/tokens (Write)
HF_USERNAME   = "Subh24ai"
WANDB_TOKEN   = ""                     # optional: wandb.ai/authorize (leave blank to skip)
# ──────────────────────────────────────────────────────────────

# Model & data
BASE_MODEL    = "Qwen/Qwen2.5-1.5B-Instruct"
DATASET_ID    = f"{HF_USERNAME}/yojana-sahayak-instruct"
OUTPUT_REPO   = f"{HF_USERNAME}/yojana-sahayak-qwen2.5-1.5b-qlora"

# LoRA hyperparameters
LORA_R        = 16
LORA_ALPHA    = 32
LORA_DROPOUT  = 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj"]

# Training
MAX_SEQ_LEN   = 512
BATCH_SIZE    = 4
GRAD_ACCUM    = 4        # effective batch = 16
LR            = 2e-4
EPOCHS        = 2
WARMUP_RATIO  = 0.05
SAVE_STEPS    = 200
EVAL_STEPS    = 200
MAX_TRAIN     = 10000    # cap training samples to fit T4 time budget (~90 min)

print(f"Base model : {BASE_MODEL}")
print(f"Dataset    : {DATASET_ID}")
print(f"Output     : {OUTPUT_REPO}")
print(f"LoRA r={LORA_R}, alpha={LORA_ALPHA}")
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")

## Step 3 — Login to HuggingFace & W&B

In [ ]:
from huggingface_hub import login
import os

login(token=HF_TOKEN)
os.environ["HF_TOKEN"] = HF_TOKEN

if WANDB_TOKEN:
    import wandb
    wandb.login(key=WANDB_TOKEN)
    os.environ["WANDB_PROJECT"] = "yojana-sahayak"
    print("W&B logging enabled")
else:
    os.environ["WANDB_DISABLED"] = "true"
    print("W&B disabled — training will still work fine")

print("Logged in to HuggingFace")

## Step 4 — Load and inspect dataset

In [ ]:
from datasets import load_dataset

ds = load_dataset(DATASET_ID)
print(f"Train : {len(ds['train']):,} examples")
print(f"Eval  : {len(ds['eval']):,} examples")
print(f"\nSample record:")
sample = ds['train'][0]
print(f"  language    : {sample['language']}")
print(f"  scheme_name : {sample['scheme_name'][:60]}")
print(f"  field       : {sample['field']}")
for msg in sample['messages']:
    role = msg['role'].upper()
    content = msg['content'][:100]
    print(f"  [{role}]: {content}")

## Step 5 — Format to chat template

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    token=HF_TOKEN
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def format_example(example):
    """Apply Qwen2.5 chat template to messages list."""
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

# Cap training set to MAX_TRAIN for time budget
train_ds = ds["train"].shuffle(seed=42).select(range(min(MAX_TRAIN, len(ds["train"]))))
eval_ds  = ds["eval"].select(range(min(500, len(ds["eval"]))))

train_formatted = train_ds.map(format_example, remove_columns=train_ds.column_names)
eval_formatted  = eval_ds.map(format_example,  remove_columns=eval_ds.column_names)

print(f"Train examples : {len(train_formatted):,}")
print(f"Eval examples  : {len(eval_formatted):,}")
print(f"\nFormatted sample (first 300 chars):")
print(train_formatted[0]["text"][:300])

## Step 6 — Load model in 4-bit (QLoRA)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # NormalFloat4 — best quality
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,     # nested quantization saves ~0.4 GB
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    token=HF_TOKEN,
)
model.config.use_cache = False          # required for gradient checkpointing
model.config.pretraining_tp = 1

# Count parameters
total = sum(p.numel() for p in model.parameters())
print(f"\nModel loaded: {total/1e9:.2f}B parameters")
print(f"VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

## Step 7 — Attach LoRA adapters

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for k-bit training (freezes base, enables grad on norm layers)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params : {trainable:,}  ({100*trainable/total:.2f}% of total)")
print(f"Total params     : {total:,}")
print(f"\nLoRA config: r={LORA_R}, alpha={LORA_ALPHA}")
print(f"Target modules   : {TARGET_MODULES}")

## Step 8 — Training

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./checkpoints",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    fp16=True,
    logging_steps=50,
    save_steps=SAVE_STEPS,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    save_total_limit=2,
    report_to="wandb" if WANDB_TOKEN else "none",
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field="text",
    packing=False,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",      # memory-efficient optimizer for QLoRA
    dataloader_num_workers=2,
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_formatted,
    eval_dataset=eval_formatted,
    processing_class=tokenizer,
)

print("Starting training...")
print(f"Steps per epoch : {len(train_formatted) // (BATCH_SIZE * GRAD_ACCUM)}")
print(f"Total steps     : {len(train_formatted) * EPOCHS // (BATCH_SIZE * GRAD_ACCUM)}")

train_result = trainer.train()

print(f"\nTraining complete!")
print(f"Train loss      : {train_result.training_loss:.4f}")
print(f"Train runtime   : {train_result.metrics['train_runtime']/60:.1f} min")

## Step 9 — Evaluate: loss + ROUGE

In [ ]:
import json
from rouge_score import rouge_scorer

# Eval loss
eval_results = trainer.evaluate()
print(f"Eval loss       : {eval_results['eval_loss']:.4f}")
print(f"Eval perplexity : {2**eval_results['eval_loss']:.2f}")

# ROUGE on 20 samples
print("\nRunning ROUGE evaluation on 20 samples...")
model.eval()
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)

rouge1_scores, rouge2_scores, rougeL_scores = [], [], []

eval_samples = ds["eval"].select(range(20))
for sample in eval_samples:
    messages = sample["messages"]
    # Build prompt (system + user, no assistant)
    prompt_messages = [m for m in messages if m["role"] != "assistant"]
    # Last assistant message is the reference
    reference = next(m["content"] for m in messages if m["role"] == "assistant")

    prompt = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    scores = scorer.score(reference, generated)
    rouge1_scores.append(scores['rouge1'].fmeasure)
    rouge2_scores.append(scores['rouge2'].fmeasure)
    rougeL_scores.append(scores['rougeL'].fmeasure)

avg_r1 = sum(rouge1_scores) / len(rouge1_scores)
avg_r2 = sum(rouge2_scores) / len(rouge2_scores)
avg_rL = sum(rougeL_scores) / len(rougeL_scores)

print(f"\nROUGE Results (20 eval samples):")
print(f"  ROUGE-1 : {avg_r1:.4f}")
print(f"  ROUGE-2 : {avg_r2:.4f}")
print(f"  ROUGE-L : {avg_rL:.4f}")
print(f"  Eval loss: {eval_results['eval_loss']:.4f}")

# Save eval results
eval_summary = {
    "model": BASE_MODEL,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "train_loss": round(train_result.training_loss, 4),
    "eval_loss": round(eval_results['eval_loss'], 4),
    "eval_perplexity": round(2**eval_results['eval_loss'], 2),
    "rouge1": round(avg_r1, 4),
    "rouge2": round(avg_r2, 4),
    "rougeL": round(avg_rL, 4),
    "train_samples": len(train_formatted),
    "train_runtime_min": round(train_result.metrics['train_runtime']/60, 1),
}
print("\nEval summary:", json.dumps(eval_summary, indent=2))

## Step 10 — Quick inference test

In [ ]:
def ask(question: str, language: str = "en") -> str:
    messages = [
        {"role": "system", "content": "You are Yojana Sahayak, a helpful assistant that provides accurate information about Indian government welfare schemes in simple, clear language. Answer in the same language the user asks in. Be concise and actionable."},
        {"role": "user",   "content": question},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

# Test English
print("[EN] Who is eligible for PM Kisan?")
print(ask("Who is eligible for PM Kisan?"))
print()

# Test Hindi
print("[HI] Ayushman Bharat mein kya milta hai?")
print(ask("Ayushman Bharat mein kya milta hai?"))
print()

# Test multi-turn
print("[HI] Mudra loan ke liye kaun apply kar sakta hai?")
print(ask("Mudra loan ke liye kaun apply kar sakta hai?"))

## Step 11 — Save adapter + push to HF Hub

In [ ]:
import json

# Save adapter locally
adapter_path = "./yojana-sahayak-adapter"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"Adapter saved to {adapter_path}")

# Write eval results into adapter folder
with open(f"{adapter_path}/eval_results.json", "w") as f:
    json.dump(eval_summary, f, indent=2)

# Push adapter to HF Hub
model.push_to_hub(OUTPUT_REPO, token=HF_TOKEN, private=False)
tokenizer.push_to_hub(OUTPUT_REPO, token=HF_TOKEN, private=False)
print(f"\nAdapter pushed to: https://huggingface.co/{OUTPUT_REPO}")

# Write model card
from huggingface_hub import HfApi
api = HfApi()

model_card = f"""---
language:
- en
- hi
license: apache-2.0
base_model: {BASE_MODEL}
tags:
- qlora
- lora
- india
- government-schemes
- hindi
- yojana
- fine-tuned
datasets:
- {DATASET_ID}
---

# Yojana Sahayak — Qwen2.5-1.5B QLoRA

Fine-tuned **{BASE_MODEL}** on Indian government scheme Q&A in English and Hindi.
Part of the [Yojana Sahayak](https://github.com/{HF_USERNAME}/yojana-sahayak) project —
a multilingual voice assistant for government welfare schemes.

## Eval Results

| Metric | Score |
|--------|-------|
| Train loss | {eval_summary['train_loss']} |
| Eval loss | {eval_summary['eval_loss']} |
| Perplexity | {eval_summary['eval_perplexity']} |
| ROUGE-1 | {eval_summary['rouge1']} |
| ROUGE-2 | {eval_summary['rouge2']} |
| ROUGE-L | {eval_summary['rougeL']} |
| Train samples | {eval_summary['train_samples']:,} |

## Training

- **Base model**: {BASE_MODEL}
- **Method**: QLoRA (4-bit NF4 + LoRA r={LORA_R}, alpha={LORA_ALPHA})
- **Dataset**: {DATASET_ID} ({eval_summary['train_samples']:,} train samples)
- **Hardware**: Kaggle T4 GPU ({eval_summary['train_runtime_min']} min)

## Usage

```python
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base = AutoModelForCausalLM.from_pretrained("{BASE_MODEL}")
model = PeftModel.from_pretrained(base, "{OUTPUT_REPO}")
tokenizer = AutoTokenizer.from_pretrained("{OUTPUT_REPO}")

messages = [
    {{"role": "system", "content": "You are Yojana Sahayak..."}},
    {{"role": "user",   "content": "PM Kisan ke liye kaun eligible hai?"}}
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt")
output = model.generate(**inputs, max_new_tokens=200)
print(tokenizer.decode(output[0], skip_special_tokens=True))
```
"""

with open("model_card.md", "w") as f:
    f.write(model_card)

api.upload_file(
    path_or_fileobj="model_card.md",
    path_in_repo="README.md",
    repo_id=OUTPUT_REPO,
    repo_type="model",
    token=HF_TOKEN,
)
print(f"Model card pushed.")
print(f"\nAll done! Model live at: https://huggingface.co/{OUTPUT_REPO}")

## Step 12 — Merge LoRA + dequantize to fp16, then push

import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel

MERGED_REPO = f"{HF_USERNAME}/yojana-sahayak-qwen2.5-1.5b-merged"

# 1. Load base model in fp16 (NOT quantized)
print("Loading base model in fp16...")
base_model_fp16 = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="cpu",
)

# 2. Load your trained LoRA adapter on top
print("Loading LoRA adapter...")
adapter_path = "./yojana-sahayak-adapter"  # local path from Step 11
peft_model = PeftModel.from_pretrained(base_model_fp16, adapter_path)

# 3. Merge LoRA into fp16 base
print("Merging...")
merged_model = peft_model.merge_and_unload()

# 4. Push clean fp16 model
print("Pushing to Hub...")
merged_model.push_to_hub(MERGED_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(MERGED_REPO, token=HF_TOKEN)
print(f"\nDone! https://huggingface.co/{MERGED_REPO}")